Build Drivers Dimension

1.Read silver drivers table

2.Read gold ref_nationality_region table

3.Join the data from drivers with ref_nationality_region using nationality

4.Select the required columns

5.Write the transformed data to gold dim_drivers ta

In [0]:
%run "/Workspace/Users/ganeshgadade157@gmail.com/Projects/Formula-f1/00.common/01.envioment_config"

In [0]:
from pyspark.sql import functions as f

# Table name for gold dim_drivers
table_name = f"{catlog_name}.{gold_schema}.dim_drivers"

# Step 1 - Read silver drivers and gold ref_nationality_region tables
df_drivers  = spark.read.table("dev.silver.drivers")
df_ref_nationality_region = spark.read.table("dev.gold.ref_nationality_region")

# Step 2 - Join drivers with nationality region and select required columns
df_dim_drivers = df_drivers.alias("d").join(
    df_ref_nationality_region.alias("n"),
    f.col("d.nationality") == f.col("n.nationality"),
    "left"
).select(
    f.col("d.driver_id"),
    f.col("d.driver_name"),
    f.col("d.nationality"),
    f.col("n.region").alias("nationality_region")
)

# Step 3 - Write to gold dim_drivers table in delta format
df_dim_drivers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(table_name)